# Exploratory Data Analysis

This notebook explores the Credit Card Fraud dataset: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
The dataset consists of credit card transactions over a course of 2 days, and each transaction has been classified as either fraudulent or nonfraudulent, represented in the Class column with 1 or 0, respectively. The Time column represents the time from the first transaction in this 2 day period, and the Amount column is how much was spent in the given transaction

Note: All features (except for the class, time, and ammount) of the dataset have undergone a PCA transformation and been anonymized into 'V' labels.

## Load and inspect data

In [1]:
import pandas as pd

file_path = '../data/creditcard.csv'
df_creditcard = pd.read_csv(file_path)

print(df_creditcard.head())
print(df_creditcard.shape)

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

## Class balance

There are very few fraudulent transactions in this dataset. This makes it difficult to use supervised learning to classify what a fraudulant transaction looks like, and it makes accuracy a less valuable metric, since classifiers that label all transactions as 'not fraud' would achieve high accuracy on this dataset. With few labeled fraud examples, the approach models what normal transactions look like and detects deviations from that

In [2]:
print(df_creditcard.Class.value_counts())
print(str(df_creditcard.Class.mean() * 100)  + '%')

Class
0    284315
1       492
Name: count, dtype: int64
0.1727485630620034%


**Finding:** Fraud rate in this dataset is just 0.173% of transactions!

## Amount Distribution: Fraud vs Normal

In [3]:
fraud = df_creditcard.loc[df_creditcard.Class == 1]
non_fraud = df_creditcard.loc[df_creditcard.Class == 0]


In [4]:
print(fraud.Amount.describe())

count     492.000000
mean      122.211321
std       256.683288
min         0.000000
25%         1.000000
50%         9.250000
75%       105.890000
max      2125.870000
Name: Amount, dtype: float64


In [5]:
print(non_fraud.Amount.describe())

count    284315.000000
mean         88.291022
std         250.105092
min           0.000000
25%           5.650000
50%          22.000000
75%          77.050000
max       25691.160000
Name: Amount, dtype: float64


**Finding:** Fraudulent transactions are significantly larger in Amount, while the standard deviation for both categories is similar.

## Correlation among V1-V28

Before moving on, it makes sense to sanity check the features, and ensure that PCA has done what it promises. The PCA transformation relies on the Spectral Theorem, which states that any real symmetric matrix, including covariance matrices, can be orthogonally diagonalized. Since the dataset has undergone the PCA transformation, the features here should be orthogonal (which implies they are uncorrelated) to each other, meaning all non-diagonal entries must be 0.

The fact that the features are uncorrelated will be important in the process of calculating the Mahalanobis distance, which requires inverting the covariance matrix. This calculation is simple only when the features are uncorrelated

In [6]:
features = df_creditcard.loc[:, ['V' + str(i) for i in range(1, 29, 1)]]
print(features.corr())

               V1            V2            V3            V4            V5  \
V1   1.000000e+00  4.188326e-16 -1.164018e-15 -9.082889e-16  2.085924e-17   
V2   4.188326e-16  1.000000e+00  3.140164e-16 -1.125909e-15  5.230686e-16   
V3  -1.164018e-15  3.140164e-16  1.000000e+00  4.644369e-16 -5.483554e-17   
V4  -9.082889e-16 -1.125909e-15  4.644369e-16  1.000000e+00 -1.722056e-15   
V5   2.085924e-17  5.230686e-16 -5.483554e-17 -1.722056e-15  1.000000e+00   
V6  -6.343731e-16  2.781752e-16  1.627805e-15 -7.565957e-16  2.208263e-16   
V7  -1.018099e-15  1.857398e-16  5.239200e-16 -4.188169e-16  2.691713e-16   
V8  -2.557889e-16 -5.698764e-17 -1.297365e-15  5.645256e-16  7.396815e-16   
V9  -1.347621e-16  2.006267e-17  5.725902e-16  6.873528e-16  7.178142e-16   
V10  7.340779e-17 -3.930237e-16  1.157736e-15  2.203905e-16 -5.150417e-16   
V11  2.248163e-16  1.965104e-16  1.603095e-15  3.498520e-16  7.178417e-16   
V12  1.872282e-16 -9.856214e-17  6.467469e-16 -5.618322e-16  7.494775e-16   

**Finding:** Upon inspection, all diagonal entries are 1, and all nondiagonal entries are (practically) 0. 


## Historical/streaming split

Splitting the dataset 70-30 into historical and streaming datasets, which will be used to fit statististics and simulate live traffic, respectively. Datasets are made to both preserve the overall fraud rate of 0.173%

In [7]:
fraud_shuffled = fraud.sample(frac=1, random_state=67).reset_index(drop=True)
non_fraud_shuffled = non_fraud.sample(frac=1, random_state=67).reset_index(drop=True)

fraud_index = int(len(fraud_shuffled) * 0.7)
non_fraud_index = int(len(non_fraud_shuffled) * 0.7)

historical = pd.concat([fraud_shuffled.iloc[:fraud_index, :], non_fraud_shuffled.iloc[:non_fraud_index, :]])
streaming = pd.concat([fraud_shuffled.iloc[fraud_index:, :], non_fraud_shuffled.iloc[non_fraud_index:, :]])

historical = historical.sample(frac=1, random_state=67).reset_index(drop=True)
streaming = streaming.sample(frac=1, random_state=67).reset_index(drop=True)

print(historical.Class.mean())
print(streaming.Class.mean())

0.0017254870488152324
0.0017321489179921118


**Finding:** historical fraud rate 0.1725%, streaming fraud rate 0.1732%

Good enough to ensure both datasets are representative of the whole

In [8]:
streaming.to_csv('../data/streaming.csv', index=False)
historical.to_csv('../data/historical.csv', index=False)

## Summary

- Dataset: 284,807 transactions, 0.173% fraud rate
- V1-V28 confirmed uncorrelated 
- Fraud transactions differ from normal transactions in their higher spending Amount
- Produced historical.csv (70%) and streaming.csv (30%), both preserving the overall fraud rate